# Exploring the IPCC AR6 Scenarios Database with pyam

**ENGG562/ENV462 · Climate Change Governance: Science, Data & Models · LUMS · Fall 2026**

**Module 2: Scenarios & Integrated Assessment** · Coding exercise 02 · Requires coding exercise 01 or equivalent experience with Python and pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LUMS-WIT/climate-futures/blob/main/content-src/coding/02-ar6-scenario-explorer-pyam.ipynb)

This notebook is a step-by-step tutorial on retrieving and analysing the scenarios that underpin the IPCC Sixth Assessment Report (AR6), Working Group III. All data are fetched directly from the [AR6 Scenario Explorer](https://data.ece.iiasa.ac.at/ar6/) hosted by IIASA, using the open-source Python package [pyam](https://pyam-iamc.readthedocs.io/en/stable/).

The notebook runs in Google Colab without any local installation. Select **Open in Colab** above, then **File → Save a copy in Drive**, and run the cells in order with **Shift + Enter**. Reading the text between the cells is as important as running the code.

Three kinds of prompt appear throughout:

- **💬 Predict before running the next cell:** write a short prediction *before* running the cell that follows, then compare it with the result.
- **✏️ Exercise:** replace every `...` or `___` with code. Self-check cells print ✅ or ❌.
- **💬 Question:** a written answer of two to four sentences.

## Learning objectives

On completing this notebook, students should be able to:

1. Explain the difference between a **model**, a **scenario**, a **variable**, and a **region** in the IAMC data format used by the IPCC.
2. Connect to the AR6 Scenarios Database programmatically and query a subset of it.
3. Use the **meta indicators** (climate category, policy category, Illustrative Mitigation Pathway) to group and compare scenarios.
4. Produce standard figures of emission pathways, warming outcomes, and energy-system change.
5. Answer quantitative policy questions with the scenario ensemble, and state the limitations of doing so.

## Structure of the notebook

| Part | Content | Approximate time |
|---|---|---|
| 0 | Setup | 5 min |
| 1 | Concepts: models, scenarios, variables, regions, meta indicators | 15 min |
| 2 | Connecting to the database and exploring what it contains | 15 min |
| 3 | First query: global CO₂ emissions | 20 min |
| 4 | Guided analysis: six policy questions | 45 min |
| 5 | In-class activity | 45–60 min |
| 6 | Homework assignment | One week |
| 7 | References and further reading | — |

## How the exercises are tiered

The activities in Parts 5 and 6 are set at two levels of programming experience and with two disciplinary emphases. Every student completes the **Core** tasks. Students then choose a **track** and a **level**.

| | **Foundation level** (limited programming experience) | **Advanced level** (prior programming experience) |
|---|---|---|
| **ENGG track** (energy and power systems emphasis) | Complete partly written code to analyse the electricity system. | Write reusable analysis functions; quantify required technology growth rates and test their sensitivity to model choice. |
| **ENV track** (land, emissions, and environmental policy emphasis) | Complete partly written code to analyse land use and non-CO₂ gases. | Write reusable analysis functions; test the robustness of land-based mitigation conclusions across model families. |

ENGG562 students are expected to attempt the ENGG track and ENV462 students the ENV track, although either track may be chosen with the instructor's agreement. The level is the student's own choice, and each level is graded against its own scope.

---
# Part 0 · Setup

The cell below installs pyam when the notebook runs in Google Colab. Installation takes roughly one minute. On a personal computer with pyam already installed (`pip install pyam-iamc`), the installation step is skipped automatically.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install --quiet pyam-iamc

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output readable for teaching purposes

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyam

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("pyam version:", pyam.__version__)

In [ ]:
def check(description, condition):
    """Self-check helper: prints a tick or a cross. It never stops the notebook."""
    try:
        passed = bool(condition())
    except Exception:
        passed = False
    print(("✅ " if passed else "❌ ") + description)

> **Troubleshooting.** The AR6 database is public and does not require an account. If a connection error mentions *credentials rejected*, an outdated IIASA login is stored on the machine; running `ixmp4 logout` in a terminal (or simply using Colab) resolves it. If a query times out, the IIASA server is briefly busy, and re-running the cell is usually sufficient.

---
# Part 1 · Concepts

## 1.1 What is an integrated assessment model?

An **integrated assessment model (IAM)** represents the energy system, the economy, land use, and (in simplified form) the climate within a single computational framework. Given assumptions about population, economic growth, technology costs, and policy, the model computes how energy would be supplied and used, how land would be allocated, and which greenhouse gases would be emitted. Examples in the AR6 database include MESSAGEix-GLOBIOM (IIASA), REMIND-MAgPIE (PIK), IMAGE (PBL), GCAM (PNNL), AIM/CGE (NIES), and WITCH (CMCC/EIEE). Lecture 9 covered the main families of such models.

## 1.2 What is a scenario?

A **scenario** is one internally consistent run of a model under one set of assumptions, for example *"SSP2 socio-economic development, with a carbon price applied from 2025 that limits cumulative emissions to 900 Gt CO₂"*. A scenario is **not a forecast**: it answers the conditional question *"what would happen if these assumptions held?"* (see Lecture 7).

The same scenario name, run through different models, usually gives different results, because each model represents the world differently. The **model–scenario pair** is therefore the unit of analysis throughout this notebook.

## 1.3 The IAMC data format

The IPCC scenario data follow a common format defined by the Integrated Assessment Modeling Consortium (IAMC). Each row is one **timeseries**, identified by five index columns, with values reported for a set of years:

| model | scenario | region | variable | unit | 2020 | 2030 | ... | 2100 |
|---|---|---|---|---|---|---|---|---|
| MESSAGEix-GLOBIOM 1.0 | LowEnergyDemand_1.3_IPCC | World | Emissions\|CO2 | Mt CO2/yr | ... | ... | ... | ... |

This is the same **wide** format practised in Section 9 of coding exercise 01.

**Variables are hierarchical.** The vertical bar `|` separates levels, so that `Primary Energy|Coal` is a component of `Primary Energy`, and `Primary Energy|Coal|w/ CCS` is a component of `Primary Energy|Coal`. This structure makes it possible to check whether components sum to their total, and to search for all variables beneath a given node.

**Regions** include `World`, the large model regions (for example the R5 and R10 groupings), and, for a smaller subset of models, individual countries.

## 1.4 Meta indicators

In addition to the timeseries, every model–scenario pair carries **meta indicators**: one value per scenario rather than per year. The most important for this course are:

- **`Category` (C1 to C8):** the climate outcome of the scenario, assessed by the IPCC with a common simple climate model (MAGICC), for example *C1: limit warming to 1.5 °C (>50%) with no or limited overshoot*. Scenarios that failed the IPCC's quality checks are labelled `failed-vetting`, and those without a climate assessment are labelled `no-climate-assessment`.
- **`Policy_category` (P0 to P3):** the type of policy assumed, ranging from no policy to immediate global action, with intermediate categories for delayed or fragmented action.
- **`IMP_marker`:** a handful of **Illustrative Mitigation Pathways** selected by the IPCC to tell distinct stories: CurPol (current policies), ModAct (moderate action), GS (gradual strengthening), Neg (heavy reliance on net-negative emissions), Ren (renewables), LD (low energy demand), and SP (shifting development pathways). Neg-2.0 and Ren-2.0 are variants of Neg and Ren that limit warming to 2 °C instead of 1.5 °C.
- **`Ssp_family`:** the Shared Socioeconomic Pathway (1 to 5) on which the scenario is based (Lecture 8).
- Several numeric indicators computed by the IPCC, such as the median peak warming and the year of net-zero CO₂.

**💬 Comprehension check.** Explain in one sentence each why (a) two models can disagree on the same scenario, and (b) a scenario in category C1 is not a prediction that warming will stay below 1.5 °C.

*Answer:* *(write the answer here)*

---
# Part 2 · Connecting to the database

`pyam.iiasa.Connection` opens a connection to one of the databases hosted by IIASA. The AR6 database is called `ar6-public`.

In [ ]:
conn = pyam.iiasa.Connection("ar6-public")

## 2.1 Which models and scenarios are available?

Each of the following calls returns a list (as a pandas Series). Querying the full lists is fast, because only names are transferred, not data.

In [ ]:
models = conn.models()
scenarios = conn.scenarios()
print(f"{len(models)} model versions and {len(scenarios)} distinct scenario names")
models.head(15)

`conn.properties()` returns one row per **model–scenario pair**, which is the number of distinct scenario runs.

**💬 Predict before running the next cell.** The next cell counts how many scenario runs each model family contributed. Will the contributions be roughly equal across families, or concentrated in a few?

*Answer:* *(write the answer here)*

In [ ]:
props = conn.properties()
print(f"{len(props)} model–scenario pairs in the database")

# Which modelling teams contributed the most runs? Model names include a version
# number, so the first word is used as an approximate "model family".
family = props.index.get_level_values("model").str.split(" ").str[0]
family.value_counts().head(10)

> **Observation.** When a few model families contribute a large share of all scenarios, a simple median across "all scenarios" gives more weight to those models. The AR6 ensemble is therefore an **ensemble of opportunity**, not a statistical sample of possible futures. Parts 5 and 6 return to this point.

## 2.2 Which variables are available?

There are more than a thousand variables. Searching by keyword is the practical way to find the relevant ones.

In [ ]:
variables = conn.variables()
print(f"{len(variables)} variables")

def search(keyword, max_depth=None):
    """Return variables containing `keyword`, optionally limited to a hierarchy depth."""
    hits = [v for v in variables if keyword.lower() in v.lower()]
    if max_depth is not None:
        hits = [v for v in hits if v.count("|") <= max_depth]
    return hits

search("Emissions|CO2", max_depth=2)

In [ ]:
# The climate outcome of every scenario was computed by the IPCC with three simple climate
# models. The MAGICC median (50th percentile) is the one used to assign categories.
search("Surface Temperature (GSAT)|MAGICC")

### ✏️ Exercise 1: Finding variables

Use `search()` to create two lists: `ch4_vars`, containing all methane emission variables up to two levels deep (search for `"Emissions|CH4"`), and `land_vars`, containing all variables that begin with `"Land Cover|"` and are exactly one level below it (`max_depth=1`).

In [ ]:
ch4_vars = ...
land_vars = ...

print(ch4_vars)
print(land_vars)

In [ ]:
check("ch4_vars contains total methane emissions", lambda: "Emissions|CH4" in ch4_vars)
check("land_vars contains forest area", lambda: "Land Cover|Forest" in land_vars)

## 2.3 Which regions are available?

In [ ]:
regions = conn.regions()
print(f"{len(regions)} regions")
[r for r in regions if "Asia" in r or r in ("World", "Pakistan", "India")]

Pakistan appears as a region, but, as Question 6 in Part 4 shows, very few models report country-level results for it. Most analyses of South Asia must rely on the R10 region *"Countries of South Asia; primarily India"*.

## 2.4 Which meta indicators are available?

In [ ]:
list(conn.meta_columns)

---
# Part 3 · First query: global CO₂ emissions

## 3.1 Querying data

`conn.query()` downloads timeseries matching the given filters and returns a `pyam.IamDataFrame`. The `meta` argument selects which meta indicators to attach. Queries for the whole ensemble can take 20 to 60 seconds.

In [ ]:
META = [
    "Category", "Category_name", "Policy_category", "IMP_marker", "Ssp_family",
    "Median peak warming (MAGICCv7.5.3)",
    "Median warming in 2100 (MAGICCv7.5.3)",
    "Year of netzero CO2 emissions (Harm-Infilled) table",
    "CO2 emissions reductions 2019-2030 % modelled Harmonized-Infilled",
    "CO2 emissions reductions 2019-2050 % modelled Harmonized-Infilled",
]

df = conn.query(
    region="World",
    variable=["Emissions|CO2", "Emissions|Kyoto Gases"],
    meta=META,
)
df

## 3.2 The structure of an IamDataFrame

An `IamDataFrame` holds two tables:

- `df.data` is the **long** format, with one row per model, scenario, region, variable, year, and value.
- `df.meta` holds **one row per model–scenario pair** with the meta indicators.

`df.timeseries()` converts the data to the **wide** format, with years as columns.

In [ ]:
df.data.head()

In [ ]:
df.timeseries().iloc[:5, -10:]

In [ ]:
df.meta.head()

## 3.3 Keeping only vetted scenarios

The IPCC assessment uses only scenarios that passed vetting and received a climate category. The `filter()` method accepts index columns, years, and meta indicators alike. Wildcards (`*`) are permitted in string filters.

In [ ]:
CATEGORIES = [f"C{i}" for i in range(1, 9)]

print("Before filtering:", len(df.meta), "scenarios")
df = df.filter(Category=CATEGORIES)
print("After filtering: ", len(df.meta), "scenarios")

df.meta["Category"].value_counts().sort_index()

In [ ]:
# The full descriptions of the categories, as stored in the database
df.meta[["Category", "Category_name"]].drop_duplicates().sort_values("Category").set_index("Category")

## 3.4 Units

Emissions are reported in megatonnes (Mt) of CO₂ per year. Gigatonnes (Gt) are easier to read, and `convert_unit()` performs the conversion.

In [ ]:
df.unit

In [ ]:
df = df.convert_unit("Mt CO2/yr", to="Gt CO2/yr").convert_unit("Mt CO2-equiv/yr", to="Gt CO2-equiv/yr")
df.unit

## 3.5 A first figure

pyam includes plotting functions that understand the IAMC format. The cell below sets a consistent colour for each climate category, which is reused in every later figure.

**💬 Predict before running the next cell.** The next figure shows global net CO₂ emissions from 2010 to 2100, grouped by category. Sketch (on paper or in words) what the C1 band and the C7 band will look like, and state in which decade the C1 band will first touch zero.

*Answer:* *(write the answer here)*

In [ ]:
# Official AR6 category colours, as stored in the database meta indicator "Category_color_hex"
CAT_COLORS = {
    "C1": "#97CEE4", "C2": "#778663", "C3": "#6F7899", "C4": "#A7C682",
    "C5": "#8CA7D0", "C6": "#FAC182", "C7": "#F18872", "C8": "#BD7161",
}
pyam.run_control().update({"color": {"Category": CAT_COLORS}})

ax = df.filter(variable="Emissions|CO2", year=range(2010, 2101)).plot(
    color="Category", fill_between=True, final_ranges=True
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Global net CO$_2$ emissions by AR6 climate category")
ax.set_ylabel("Gt CO$_2$/yr")
plt.show()

The shaded bands show the range (by default the minimum to maximum) of all scenarios in each category, and the bars on the right show the spread in 2100.

**💬 Comprehension check.** Why does the band for C1 cross zero earlier than the band for C3? Do all C3 scenarios appear to reach net-zero CO₂ by 2100? (Question 2 in Part 4 checks this with numbers.)

*Answer:* *(write the answer here)*

## 3.6 Selecting individual scenarios

The Illustrative Mitigation Pathways are a compact way to compare distinct narratives. `keep=False` inverts a filter: here it keeps every scenario whose IMP marker is **not** `non-IMP`.

In [ ]:
imps = df.filter(IMP_marker="non-IMP", keep=False).filter(variable="Emissions|CO2")
imps.meta[["IMP_marker", "Category", "Policy_category"]].sort_values("Category")

In [ ]:
ax = imps.filter(year=range(2010, 2101)).plot(color="IMP_marker", legend=dict(loc="center left", bbox_to_anchor=(1, 0.5)))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Global net CO$_2$ emissions in the Illustrative Mitigation Pathways")
ax.set_ylabel("Gt CO$_2$/yr")
plt.show()

### ✏️ Exercise 2: Filtering by meta indicator

**(a)** Filter `df` to the IMP with marker `"LD"` (low energy demand) and store it in `ld`. Print its model, scenario, category, and policy category.

**(b)** Repeat the figure from Section 3.5, but for **all greenhouse gases** (`Emissions|Kyoto Gases`) rather than CO₂. Label the y-axis with the correct unit.

In [ ]:
# (a)
ld = df.filter(IMP_marker=___)
print(ld.meta[["Category", "Policy_category"]])

# (b)
ax = df.filter(variable=___, year=range(2010, 2101)).plot(color="Category", fill_between=True)
...
plt.show()

In [ ]:
check("ld contains exactly one scenario", lambda: len(ld.meta) == 1)

**💬 Question.** Compare the two figures. Do total greenhouse gas emissions fall below zero as often as CO₂ emissions do? What explains the difference?

*Answer:* *(write the answer here)*

---
# Part 4 · Guided analysis: six policy questions

Each question below is answered with the database. The code is complete and commented; the task while working through it is to understand each step well enough to modify it in Parts 5 and 6.

To avoid repeated downloads, a single larger query is made here for all the variables used in this part.

In [ ]:
VARIABLES = [
    # emissions
    "Emissions|CO2", "Emissions|Kyoto Gases", "Emissions|CH4",
    "Emissions|CO2|Energy and Industrial Processes", "Emissions|CO2|AFOLU",
    # climate outcome
    "AR6 climate diagnostics|Surface Temperature (GSAT)|MAGICCv7.5.3|50.0th Percentile",
    # energy
    "Primary Energy", "Primary Energy|Coal", "Primary Energy|Oil", "Primary Energy|Gas",
    "Primary Energy|Nuclear", "Primary Energy|Biomass", "Primary Energy|Hydro",
    "Primary Energy|Solar", "Primary Energy|Wind",
    "Secondary Energy|Electricity", "Secondary Energy|Electricity|Solar",
    "Secondary Energy|Electricity|Wind", "Secondary Energy|Electricity|Coal",
    "Final Energy", "Final Energy|Electricity",
    # carbon dioxide removal
    "Carbon Sequestration|CCS|Biomass", "Carbon Sequestration|Direct Air Capture",
    # economy and land
    "Price|Carbon", "Land Cover|Forest", "Land Cover|Cropland",
]

data = conn.query(region="World", variable=VARIABLES, meta=META)
data = data.filter(Category=CATEGORIES)
data = data.convert_unit("Mt CO2/yr", to="Gt CO2/yr")
data

> **Saving a local copy.** The queried data can be stored with `data.to_excel("ar6_world.xlsx")` and reloaded later with `pyam.IamDataFrame("ar6_world.xlsx")`. The Excel format keeps both the timeseries and the meta indicators. This avoids repeating the download, and also documents exactly which data were used in an analysis.

## Question 1 · How much warming does each category imply?

The meta indicator `Median peak warming (MAGICCv7.5.3)` gives the peak of the median warming projection for each scenario. A box plot by category summarises the distribution.

**💬 Predict before running the next cell.** For which category will the gap between **peak** warming and warming **in 2100** be largest? Use the category definitions from Section 3.3.

*Answer:* *(write the answer here)*

In [ ]:
meta = data.meta.copy()
meta["Peak warming (°C)"] = pd.to_numeric(meta["Median peak warming (MAGICCv7.5.3)"], errors="coerce")
meta["Warming in 2100 (°C)"] = pd.to_numeric(meta["Median warming in 2100 (MAGICCv7.5.3)"], errors="coerce")

fig, ax = plt.subplots()
meta.boxplot(column="Peak warming (°C)", by="Category", ax=ax, grid=False)
ax.axhline(1.5, color="grey", linestyle="--", linewidth=0.8)
ax.axhline(2.0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Median peak warming of each scenario, by category")
ax.set_ylabel("°C above 1850–1900")
fig.suptitle("")
plt.show()

meta.groupby("Category")[["Peak warming (°C)", "Warming in 2100 (°C)"]].median().round(2)

The full temperature trajectory over time is also available as a variable.

In [ ]:
temp = data.filter(variable="AR6 climate diagnostics|Surface Temperature (GSAT)|MAGICCv7.5.3|50.0th Percentile")
ax = temp.filter(year=range(2000, 2101)).plot(color="Category", fill_between=True)
ax.set_title("Median global surface temperature projection (MAGICC)")
ax.set_ylabel("°C above 1850–1900")
plt.show()

## Question 2 · When must global CO₂ emissions reach net zero?

The year of net-zero CO₂ is computed here directly from the timeseries: for every scenario, the first reported year in which net CO₂ emissions are zero or negative. Because most models report only every five or ten years, this is an approximation.

In [ ]:
co2 = data.filter(variable="Emissions|CO2", year=range(2020, 2101)).timeseries()
co2.index = co2.index.droplevel(["region", "variable", "unit"])

def first_year_at_or_below_zero(row):
    below = row[row <= 0]
    return below.index.min() if len(below) else np.nan

nz = co2.apply(first_year_at_or_below_zero, axis=1).rename("Net-zero CO2 year (computed)")
nz = nz.to_frame().join(data.meta[["Category", "Year of netzero CO2 emissions (Harm-Infilled) table"]])
nz.head()

In [ ]:
summary = nz.groupby("Category")["Net-zero CO2 year (computed)"].describe(percentiles=[0.05, 0.5, 0.95])
summary[["count", "5%", "50%", "95%"]]

The `count` column is lower than the number of scenarios in a category when some of its scenarios never reach net-zero CO₂ before 2100.

**💬 Question.** Compare the median net-zero year for C1 with the net-zero year obtained from the simple linear carbon budget model in Section 10 of coding exercise 01. Then compare the computed net-zero year with the IPCC's own value in `nz`, which was computed from **harmonised** emissions (adjusted so that every model starts from the same historical value). Why might the two differ, and which is more appropriate for comparing scenarios across models?

*Answer:* *(write the answer here)*

## Question 3 · How does the global energy mix change?

A stacked area chart shows the composition of primary energy in one scenario. The IMP *SP* (shifting pathways) and *CurPol* (current policies) are compared.

**💬 Predict before running the next cell.** Name two differences between the CurPol and SP energy mixes in 2050 that the next figure will show.

*Answer:* *(write the answer here)*

In [ ]:
PE_COMPONENTS = [
    "Primary Energy|Coal", "Primary Energy|Oil", "Primary Energy|Gas", "Primary Energy|Nuclear",
    "Primary Energy|Biomass", "Primary Energy|Hydro", "Primary Energy|Solar", "Primary Energy|Wind",
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, marker in zip(axes, ["CurPol", "SP"]):
    sel = data.filter(IMP_marker=marker, variable=PE_COMPONENTS, year=range(2010, 2101))
    sel.plot.stack(ax=ax, stack="variable", total=True, legend=(marker == "SP"))
    model, scenario = sel.meta.index[0]
    ax.set_title(f"{marker}: {model}\n{scenario}", fontsize=10)
    ax.set_ylabel("EJ/yr")
axes[1].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

Lecture 10 discussed how primary energy from solar and wind is accounted for differently from fossil fuels, which affects how the size of the renewable wedges in these charts should be interpreted.

## Question 4 · How quickly does coal decline, and how quickly do solar and wind grow?

The ensemble median for each category is computed from the long-format data with pandas. The function `category_median()` is reused in Parts 5 and 6.

In [ ]:
def category_median(df, variable, years):
    """Median across scenarios of one variable, by category, for selected years."""
    sub = df.filter(variable=variable, year=years)
    long = sub.data.merge(sub.meta[["Category"]], left_on=["model", "scenario"], right_index=True)
    return long.pivot_table(index="Category", columns="year", values="value", aggfunc="median")

YEARS = [2020, 2030, 2040, 2050]
print("Primary energy from coal (EJ/yr), median by category")
category_median(data, "Primary Energy|Coal", YEARS).round(0)

In [ ]:
# Variables can be combined arithmetically. Here, solar and wind electricity are added
# and divided by total electricity generation to give a share.
# With append=True, each new variable is added to `elec` in place.
elec = data.filter(variable=["Secondary Energy|Electricity", "Secondary Energy|Electricity|Solar",
                             "Secondary Energy|Electricity|Wind"])
elec.add("Secondary Energy|Electricity|Solar", "Secondary Energy|Electricity|Wind",
         "Solar and wind electricity", append=True)
elec.divide("Solar and wind electricity", "Secondary Energy|Electricity",
            "Solar and wind share of electricity", ignore_units="-", append=True)
elec.multiply("Solar and wind share of electricity", 100,
              "Solar and wind share of electricity (%)", ignore_units="%", append=True)

print("Solar and wind share of global electricity generation (%), median by category")
category_median(elec, "Solar and wind share of electricity (%)", YEARS).round(0)

### ✏️ Exercise 3: Reusing a function

Use `category_median()` to produce the same table for **natural gas** (`Primary Energy|Gas`) and store it in `gas_table`. Is the decline of gas in C1 faster or slower than the decline of coal?

In [ ]:
gas_table = category_median(data, ___, YEARS)
gas_table.round(0)

In [ ]:
check("gas_table has one row per category", lambda: set(gas_table.index) <= set(CATEGORIES) and len(gas_table) >= 7)

## Question 5 · How much carbon dioxide removal do the lowest-warming pathways rely on?

Carbon dioxide removal (CDR) through bioenergy with carbon capture and storage (BECCS) and direct air capture (DAC) is one of the most debated assumptions in mitigation scenarios. Not every model represents DAC, so missing DAC values are treated as zero only where the model reported BECCS.

**💬 Predict before running the next cell.** Will the amount of BECCS plus DAC in 2050 be similar across all C1 scenarios, or will it vary widely?

*Answer:* *(write the answer here)*

In [ ]:
cdr = data.filter(variable=["Carbon Sequestration|CCS|Biomass", "Carbon Sequestration|Direct Air Capture"],
                  year=2050).convert_unit("Mt CO2/yr", to="Gt CO2/yr")
cdr_wide = cdr.data.pivot_table(index=["model", "scenario"], columns="variable", values="value")
cdr_wide = cdr_wide.dropna(subset=["Carbon Sequestration|CCS|Biomass"]).fillna(0)
cdr_wide["BECCS + DAC in 2050 (Gt CO2/yr)"] = cdr_wide.sum(axis=1)
cdr_wide = cdr_wide.join(data.meta["Category"])

fig, ax = plt.subplots()
cdr_wide.boxplot(column="BECCS + DAC in 2050 (Gt CO2/yr)", by="Category", ax=ax, grid=False)
ax.set_title("Carbon dioxide removal by BECCS and DAC in 2050")
ax.set_ylabel("Gt CO$_2$/yr")
fig.suptitle("")
plt.show()

**💬 Discussion.** A policymaker is told that "limiting warming to 1.5 °C requires large-scale carbon removal". Using the figure, explain whether the AR6 database supports that statement as written, and how it should be qualified.

*Answer:* *(write the answer here)*

## Question 6 · What does the database say about South Asia and Pakistan?

Country-level coverage is uneven. The cell below counts how many model–scenario pairs report key variables for Pakistan, for the R10 South Asia region, and for India.

**💬 Predict before running the next cell.** Roughly how many model–scenario pairs will report CO₂ emissions for Pakistan: none, a handful, a few dozen, or several hundred?

*Answer:* *(write the answer here)*

In [ ]:
REGIONS = ["Pakistan", "Countries of South Asia; primarily India", "India"]
coverage = conn.query(region=REGIONS, variable=["Emissions|CO2", "Primary Energy", "Population"])

coverage_table = (
    coverage.data.groupby(["region", "variable"])
    .apply(lambda g: g[["model", "scenario"]].drop_duplicates().shape[0])
    .unstack("variable")
)
print("Number of model–scenario pairs reporting each variable")
coverage_table

In [ ]:
print("Models reporting Pakistan:", coverage.filter(region="Pakistan").model)

National analysis for Pakistan therefore usually starts from the R10 South Asia region and is complemented with national models or downscaling methods, a point that returns in Module 4 and in the course project.

The cell below shows CO₂ emissions for South Asia by category. The categories were assigned on the basis of **global** warming outcomes, so this figure shows *what South Asia's emissions look like in scenarios that achieve a given global outcome*, not a regional target.

In [ ]:
sa = conn.query(region="Countries of South Asia; primarily India",
                variable=["Emissions|CO2", "Emissions|CH4", "Emissions|CO2|AFOLU",
                          "Primary Energy|Coal", "Secondary Energy|Electricity",
                          "Secondary Energy|Electricity|Solar", "Secondary Energy|Electricity|Wind"],
                meta=["Category"])
sa = sa.filter(Category=CATEGORIES).convert_unit("Mt CO2/yr", to="Gt CO2/yr")

ax = sa.filter(variable="Emissions|CO2", year=range(2010, 2101)).plot(color="Category", fill_between=True)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Net CO$_2$ emissions, South Asia (R10), by global climate category")
ax.set_ylabel("Gt CO$_2$/yr")
plt.show()

**💬 Question.** Global CO₂ in C1 falls steeply from the 2020s. Does South Asia's CO₂ in C1 follow the same shape? Give one economic and one equity reason why a regional pathway might differ from the global one.

*Answer:* *(write the answer here)*

---
# Part 5 · In-class activity

**Format.** Groups of three, 45 to 60 minutes, followed by a short report-back. Each group submits one completed copy of this notebook at the end of class, with group members' names in the cell below.

**Theme: what does "Paris-consistent" require between now and 2050?**

All groups complete Task 5.1 (Core). Each group then chooses one track (5.2 ENGG or 5.3 ENV) at the Foundation or Advanced level. Groups that finish the Foundation level may continue with the Advanced level of their track.

The data loaded in Part 4 (`data`, `elec`, `cdr_wide`, and `sa`) are reused here, so Parts 0 to 4 must be run first. In Foundation-level cells, replace each `___` with a variable name, number, or short expression.

**Group members:** *(replace this text)*

**Track and level chosen:** *(for example, ENV track, Foundation level)*

## 5.1 Core task (all groups)

### ✏️ Exercise 4: Emission reductions by category

**(a)** Use `category_median()` to produce a table `t` of median global net CO₂ emissions (`Emissions|CO2`) in 2020, 2030, 2040, and 2050 for every category.

**(b)** Add two columns to `t`: the percentage reduction between 2020 and 2030 (`"cut_2030_%"`) and between 2020 and 2050 (`"cut_2050_%"`).

**(c)** Compute the median by category of the IPCC meta indicators `CO2 emissions reductions 2019-2030 % modelled Harmonized-Infilled` and `CO2 emissions reductions 2019-2050 % modelled Harmonized-Infilled`, and store them in `ipcc_cuts`.

In [ ]:
# (a)
t = category_median(data, ___, [2020, 2030, 2040, 2050])

# (b)
t["cut_2030_%"] = 100 * (1 - t[___] / t[___])
t["cut_2050_%"] = ___
print(t.round(1))

# (c)
cols = ["CO2 emissions reductions 2019-2030 % modelled Harmonized-Infilled",
        "CO2 emissions reductions 2019-2050 % modelled Harmonized-Infilled"]
ipcc_cuts = data.meta[cols].apply(pd.to_numeric, errors="coerce").groupby(data.meta["Category"]).___()
ipcc_cuts.round(1)

In [ ]:
check("t has the two reduction columns", lambda: {"cut_2030_%", "cut_2050_%"} <= set(t.columns))
check("C1 cuts more by 2050 than C7", lambda: t.loc["C1", "cut_2050_%"] > t.loc["C7", "cut_2050_%"])
check("ipcc_cuts has one row per category", lambda: len(ipcc_cuts) == 8)

**💬 Question.** Compare the reductions in `t` with `ipcc_cuts` and with Table SPM.2 of the AR6 WGIII Summary for Policymakers. Identify two reasons why the numbers do not match exactly.

*Answer:* *(write the answer here)*

## 5.2 ENGG track: electricity-system transformation

### ✏️ Exercise 5: Electrification of final energy · ENGG · Foundation

Compute the **electrification share of final energy** (`Final Energy|Electricity` divided by `Final Energy`, in percent) and report the median by category for 2020, 2030, and 2050 in a table `electrification`.

In [ ]:
fe = data.filter(variable=["Final Energy", "Final Energy|Electricity"])
fe.divide(___, ___, "Electrification share", ignore_units="-", append=True)
fe.multiply("Electrification share", 100, "Electrification share (%)", ignore_units="%", append=True)

electrification = category_median(fe, ___, [2020, 2030, 2050])
electrification.round(0)

In [ ]:
check("shares are between 0 and 100", lambda: electrification.min().min() > 0 and electrification.max().max() < 100)
check("C1 electrifies more by 2050 than C7", lambda: electrification.loc["C1", 2050] > electrification.loc["C7", 2050])

### ✏️ Exercise 6: Coal power in 2030 and 2040 · ENGG · Foundation

Produce side-by-side box plots of **coal electricity generation** (`Secondary Energy|Electricity|Coal`, EJ/yr) in 2030 and 2040 for C1, C3, and C7.

In [ ]:
coal = data.filter(variable=___, year=[2030, 2040], Category=["C1", "C3", "C7"])
long = coal.data.merge(coal.meta[["Category"]], left_on=["model", "scenario"], right_index=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, yr in zip(axes, [2030, 2040]):
    long[long["year"] == ___].boxplot(column="value", by="Category", ax=ax, grid=False)
    ax.set_title(str(yr))
axes[0].set_ylabel(___)
fig.suptitle("")
plt.show()

**💬 Question.** In three to five sentences, state what these results imply for a country whose electricity system is planning new coal capacity with a 30-year operating life.

*Answer:* *(write the answer here)*

### ✏️ Exercise 7: Required growth rates for solar and wind · ENGG · Advanced

**(a)** Write a function `cagr(df, variable, start, end)` that returns the compound annual growth rate (in percent) of a variable between two years for every model–scenario pair, as a pandas Series. Use it on `elec` to compute the growth rate of `"Solar and wind electricity"` between 2020 and 2030, and report the median for C1 and C3.

**(b)** The ensemble over-represents some model families. Recompute the C1 and C3 medians by first taking the median **within** each model family and then the median **across** families. Report whether the conclusion changes.

**(c)** Compare the result with the historical growth rate of global solar and wind generation over the preceding decade, citing the source used (for example Ember, IEA, or IRENA statistics).

In [ ]:
def cagr(df, variable, start, end):
    """Compound annual growth rate (%) of `variable` between `start` and `end`, per model–scenario."""
    ...

growth = ...

## 5.3 ENV track: land use and non-CO₂ gases

### ✏️ Exercise 8: Land-use CO₂ · ENV · Foundation

Compute the median **CO₂ emissions from agriculture, forestry, and other land use** (`Emissions|CO2|AFOLU`, Gt CO₂/yr) by category for 2020, 2030, and 2050, in a table `afolu`. Count in how many categories the 2050 median is negative (a net sink) and store the count in `n_sink`.

In [ ]:
afolu = category_median(data, ___, [2020, 2030, 2050])
n_sink = (afolu[___] < 0).sum()

print(afolu.round(2))
print("Categories in which AFOLU is a net sink in 2050:", n_sink)

In [ ]:
check("afolu has one row per category", lambda: len(afolu) == 8)

### ✏️ Exercise 9: Methane · ENV · Foundation

Compute the median **methane emissions** (`Emissions|CH4`, Mt CH₄/yr) by category for 2020 and 2050 in a table `ch4`, and add a column `"cut_2050_%"` with the percentage reduction. Compare the reductions for C1, C3, and C7 with the CO₂ reductions from Task 5.1.

In [ ]:
ch4 = category_median(data, ___, [2020, 2050])
ch4["cut_2050_%"] = ___
ch4.round(1)

In [ ]:
check("C1 cuts methane more than C7", lambda: ch4.loc["C1", "cut_2050_%"] > ch4.loc["C7", "cut_2050_%"])

**💬 Question.** In three to five sentences, discuss what these results imply for an agricultural economy with a large livestock sector, such as Pakistan.

*Answer:* *(write the answer here)*

### ✏️ Exercise 10: Land-use trade-offs and bioenergy · ENV · Advanced

**(a)** Compute the change in **forest area** and **cropland area** (`Land Cover|Forest`, `Land Cover|Cropland`, million ha) between 2020 and 2050 for every scenario, and plot forest change against cropland change, coloured by category.

**(b)** Using `cdr_wide` from Question 5, test whether scenarios with more BECCS in 2050 show larger cropland expansion. Quantify the relationship with a Spearman rank correlation, for all scenarios and within the two largest model families.

**(c)** Assess whether the conclusion holds when each model family is weighted equally, and discuss what this means for claims about land-based mitigation in national policy documents.

In [ ]:
# (a)
...

# (b)
...

---
# Part 6 · Homework assignment

**Title:** A scenario brief for a South Asian decision-maker

**Deliverables:** (1) the completed notebook (`.ipynb`), runnable from top to bottom in Colab; (2) a brief of no more than 800 words with two to four figures, submitted as a PDF. The due date is announced in class. The assignment is evaluated under the Assignments rubric on the course [Grading page](https://lums-wit.github.io/climate-futures/grading.html).

**Use of AI tools.** In line with the course policy, AI tools may be used for this assignment on the condition that every prompt used is submitted alongside the work, in the final cell of the notebook. AI-assisted and non-AI-assisted submissions are evaluated against different bars.

## Task

The brief is written for a senior official in a South Asian ministry (energy, climate, or planning) who has asked: *"What do the IPCC scenarios tell us about the changes our region would need to make between now and 2050 if the world meets the Paris Agreement goals?"* The brief should answer that question with evidence from the AR6 database, and should state clearly what the database **cannot** answer.

All submissions must:

1. Use the R10 region *Countries of South Asia; primarily India* as the main region, and state the coverage limitation for Pakistan with the numbers from Question 6.
2. Compare at least three categories, one of which must be C1 and one of which must be C7 or C8.
3. Include at least one figure that shows the **range** across scenarios, not only a median.
4. State at least two limitations of the scenario ensemble that affect the conclusions.

## Track-specific requirements

| | **Foundation level** | **Advanced level** |
|---|---|---|
| **ENGG track** | Analyse primary energy by source and the electricity generation mix for South Asia. Report the change in coal, gas, solar, and wind between 2020 and 2050 by category. | As Foundation, and additionally: compute required annual generation growth rates for solar and wind; test the sensitivity of the conclusions to model family; and identify which assumptions (for example technology costs or demand growth) most plausibly drive the spread, with reference to model documentation. |
| **ENV track** | Analyse CO₂ from AFOLU, methane, and land cover for South Asia. Report the change in each between 2020 and 2050 by category. | As Foundation, and additionally: relate land-use change to bioenergy and BECCS deployment in the same scenarios; test the sensitivity to model family; and discuss the implications for food security and water demand, with reference to at least two peer-reviewed sources. |

## Guiding questions for the brief

Answering these questions in order produces a well-structured brief:

1. What does the region's energy system (ENGG) or land system (ENV) look like in 2020 according to the models, and how does that compare with recent observed data?
2. How large and how fast are the changes by 2050 in C1 compared with C7 or C8?
3. How much do the models disagree, and is the disagreement larger for some variables than for others?
4. Which of the changes would be hardest to achieve in practice, and why?
5. What would a decision-maker need to know that the database cannot tell them?

## Guidance

- Start from the code in Part 4 and the functions written in Part 5; most of the analysis is a matter of changing the region and the variable names.
- Use `search()` from Part 2 to find the exact variable names before querying.
- Query only the variables needed; a smaller query is faster and less likely to time out.
- Not every model reports every variable. Report the number of scenarios behind each statistic.
- The brief is a policy document, not a description of code. Each figure needs a caption that states what it shows and what follows from it.

In [ ]:
# Homework: begin the analysis here. Add as many cells as needed.

**Prompts used (if any AI tool was used):** *(paste every prompt here, in order)*

---
# Part 7 · References and further reading

**Data**

- Byers, E., Krey, V., Kriegler, E., Riahi, K., Schaeffer, R., Kikstra, J., et al. (2022). *AR6 Scenarios Database.* Integrated Assessment Modeling Consortium and International Institute for Applied Systems Analysis. [doi:10.5281/zenodo.5886911](https://doi.org/10.5281/zenodo.5886911). Any use of these data in coursework or publications should cite this source.
- [AR6 Scenario Explorer](https://data.ece.iiasa.ac.at/ar6/) (web interface to the same database, useful for browsing before writing code).

**Software**

- Huppmann, D., Gidden, M. J., Nicholls, Z., Hörsch, J., Lamboll, R., Kishimoto, P. N., et al. (2021). pyam: Analysis and visualisation of integrated assessment and macro-energy scenarios. *Open Research Europe*, 1:74. [doi:10.12688/openreseurope.13633.2](https://doi.org/10.12688/openreseurope.13633.2)
- [pyam documentation](https://pyam-iamc.readthedocs.io/en/stable/), in particular the tutorials on querying IIASA databases and on plotting.

**Assessment**

- IPCC (2022). [Summary for Policymakers](https://www.ipcc.ch/report/ar6/wg3/chapter/summary-for-policymakers/). In: *Climate Change 2022: Mitigation of Climate Change.* Contribution of Working Group III to the Sixth Assessment Report of the Intergovernmental Panel on Climate Change. See Box SPM.1 on scenarios and Table SPM.2.
- Riahi, K., Schaeffer, R., et al. (2022). [Chapter 3: Mitigation Pathways Compatible with Long-term Goals](https://www.ipcc.ch/report/ar6/wg3/chapter/chapter-3/). In: *Climate Change 2022: Mitigation of Climate Change.* The chapter defines the climate categories and the Illustrative Mitigation Pathways used in this notebook.